# Instagram Data Analysis, Engineering & Data Science Project

**Business goal:** Analyze Instagram posts and engagement to identify strong content types, hashtag patterns, follower-growth signals, and an evidence-based posting plan for Alfido Tech.

**Deliverables created by this notebook**
- Clean relational SQLite database (`instagram_analytics.db`)
- Reusable post-level analytical table (`post_analytics.csv`)
- Descriptive and diagnostic analysis with visuals
- Predictive model for high-engagement posts and interpretable feature importance
- Optimal content calendar and five practical growth strategies

> **Important evidence limitation:** All post, like, comment, follow, and tag event timestamps in the supplied data are the same. Therefore, the dataset cannot honestly identify a best weekday or hour. The notebook detects this limitation instead of inventing time-based results and proposes a test calendar for future measurement.

## 1. Imports and reproducibility

In [1]:
from pathlib import Path
import json, re, sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 100)

OUTPUT_DIR = Path('instagram_project_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Output directory:', OUTPUT_DIR.resolve())

Output directory: C:\Users\Kartik\Downloads\instagram_project_outputs


## 2. Load all source tables

The loader works both with the uploaded workspace and with Kaggle. Each CSV represents one business entity or event: users, posts, likes, comments, follows, tags, and the bridge between posts and tags.

In [ ]:
candidate_dirs = [Path('upload'), Path('/kaggle/input/instgram'), Path('/kaggle/input/instagram')]
DATA_DIR = next((p for p in candidate_dirs if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Place the seven CSV files in upload/ or a Kaggle input folder.')

files = {
    'users': 'users.csv', 'photos': 'photos.csv', 'likes': 'likes.csv',
    'comments': 'comments(4).csv', 'follows': 'follows.csv',
    'photo_tags': 'photo_tags.csv', 'tags': 'tags.csv'
}
raw = {name: pd.read_csv(DATA_DIR / filename) for name, filename in files.items()}
pd.DataFrame([{'table': k, 'rows': len(v), 'columns': v.shape[1]} for k,v in raw.items()])

## 3. Data engineering: standardize schema and types

Column names are converted to `snake_case`; inconsistent category spelling/case is normalized; identifiers become nullable integers; and dates are parsed explicitly as day-first timestamps. These steps make SQL joins repeatable and prevent silent key mismatches.

In [ ]:
def snake(s):
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', s.strip().lower())).strip('_')

dfs = {}
for name, df in raw.items():
    x = df.copy()
    x.columns = [snake(c) for c in x.columns]
    dfs[name] = x

users = dfs['users'].rename(columns={'id':'user_id','private_public':'is_private','created_time':'user_created_at'})
photos = dfs['photos'].rename(columns={'id':'photo_id','user_id':'owner_user_id','created_dat':'photo_created_at','insta_filter_used':'filter_used','image_link':'image_url'})
likes = dfs['likes'].rename(columns={'user':'user_id','photo':'photo_id','created_time':'like_created_at','following_or_not':'is_following'})
comments = dfs['comments'].rename(columns={'id':'comment_id','user_id':'user_id','photo_id':'photo_id','created_timestamp':'comment_created_at','hashtags_used_count':'comment_hashtag_count'})
follows = dfs['follows'].rename(columns={'followee':'followee_id','created_time':'follow_created_at','followee_acc_status':'followee_account_status','is_follower_active':'is_active'})
photo_tags = dfs['photo_tags'].rename(columns={'photo':'photo_id','tag_id':'tag_id','user_id':'tagged_user_id'})
tags = dfs['tags'].rename(columns={'id':'tag_id','tag_text':'tag_name','created_time':'tag_created_at'})

def yes_no(series):
    return series.astype(str).str.strip().str.lower().map({'yes':1,'no':0})
for df, cols in [(users,['is_private','verified_status']), (photos,['filter_used']), (likes,['is_following']), (comments,['emoji_used'])]:
    for c in cols: df[c] = yes_no(df[c]).astype('Int64')
follows['followee_account_status'] = follows['followee_account_status'].astype(str).str.strip().str.lower()
follows['is_active'] = pd.to_numeric(follows['is_active'], errors='coerce').astype('Int64')

for df, cols in [(users,['user_id']), (photos,['photo_id','owner_user_id']), (likes,['user_id','photo_id']),
                 (comments,['comment_id','user_id','photo_id']), (follows,['follower','followee_id']),
                 (photo_tags,['photo_id','tag_id','tagged_user_id']), (tags,['tag_id'])]:
    for c in cols: df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')

for df, col in [(users,'user_created_at'),(photos,'photo_created_at'),(likes,'like_created_at'),
                (comments,'comment_created_at'),(follows,'follow_created_at'),(tags,'tag_created_at')]:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

tables = {'users':users,'photos':photos,'likes':likes,'comments':comments,'follows':follows,'photo_tags':photo_tags,'tags':tags}
display(pd.DataFrame({'rows':{k:len(v) for k,v in tables.items()}, 'duplicate_rows':{k:int(v.duplicated().sum()) for k,v in tables.items()}, 'missing_cells':{k:int(v.isna().sum().sum()) for k,v in tables.items()}}))

## 4. Data-quality and relationship checks

In [ ]:
quality = {
    'orphan_like_photo_ids': int((~likes.photo_id.isin(photos.photo_id)).sum()),
    'orphan_comment_photo_ids': int((~comments.photo_id.isin(photos.photo_id)).sum()),
    'orphan_photo_owner_ids': int((~photos.owner_user_id.isin(users.user_id)).sum()),
    'orphan_photo_tag_ids': int((~photo_tags.tag_id.isin(tags.tag_id)).sum()),
    'duplicate_like_pairs': int(likes.duplicated(['user_id','photo_id']).sum()),
    'unique_post_timestamps': int(photos.photo_created_at.nunique()),
    'unique_like_timestamps': int(likes.like_created_at.nunique()),
    'unique_comment_timestamps': int(comments.comment_created_at.nunique())
}
display(pd.Series(quality, name='value').to_frame())
if quality['unique_post_timestamps'] < 2:
    display(Markdown('**Time-analysis warning:** The source has only one post timestamp, so weekday/hour rankings are not statistically identifiable.'))

## 5. Build a normalized SQLite database

Unlike concatenating unrelated events into one sparse table, this design preserves the natural grain of each entity. Foreign-key indexes accelerate joins, while SQL views create a safe post-level semantic layer.

In [ ]:
DB_PATH = OUTPUT_DIR / 'instagram_analytics.db'
if DB_PATH.exists(): DB_PATH.unlink()
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON')
for name, df in tables.items():
    export = df.copy()
    for c in export.select_dtypes(include=['datetime64[ns]']).columns:
        export[c] = export[c].dt.strftime('%Y-%m-%d %H:%M:%S')
    export.to_sql(name, conn, index=False, if_exists='replace')

for sql in [
    'CREATE INDEX idx_photos_owner ON photos(owner_user_id)',
    'CREATE INDEX idx_likes_photo ON likes(photo_id)',
    'CREATE INDEX idx_comments_photo ON comments(photo_id)',
    'CREATE INDEX idx_follows_followee ON follows(followee_id)',
    'CREATE INDEX idx_photo_tags_photo ON photo_tags(photo_id)',
    'CREATE INDEX idx_photo_tags_tag ON photo_tags(tag_id)']:
    conn.execute(sql)
conn.commit()
pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type IN ('table','index') ORDER BY type, name", conn)

## 6. SQL feature engineering: one row per post

In [ ]:
post_sql = '''
WITH like_stats AS (
 SELECT photo_id, COUNT(*) AS likes, COUNT(DISTINCT user_id) AS unique_likers,
        AVG(COALESCE(is_following,0)) AS follower_like_share
 FROM likes GROUP BY photo_id
), comment_stats AS (
 SELECT photo_id, COUNT(*) AS comments, COUNT(DISTINCT user_id) AS unique_commenters,
        AVG(COALESCE(emoji_used,0)) AS emoji_comment_share,
        AVG(COALESCE(comment_hashtag_count,0)) AS avg_comment_hashtags
 FROM comments GROUP BY photo_id
), tag_stats AS (
 SELECT pt.photo_id, COUNT(DISTINCT pt.tag_id) AS tag_count,
        GROUP_CONCAT(DISTINCT t.tag_name) AS hashtags
 FROM photo_tags pt LEFT JOIN tags t ON t.tag_id=pt.tag_id GROUP BY pt.photo_id
), follower_stats AS (
 SELECT followee_id AS user_id, COUNT(DISTINCT follower) AS followers,
        SUM(COALESCE(is_active,0)) AS active_followers
 FROM follows GROUP BY followee_id
)
SELECT p.photo_id, p.owner_user_id, p.photo_created_at, p.photo_type, p.filter_used,
       u.is_private, u.verified_status, u.post_count,
       COALESCE(fs.followers,0) AS owner_followers,
       COALESCE(fs.active_followers,0) AS owner_active_followers,
       COALESCE(ls.likes,0) AS likes, COALESCE(ls.unique_likers,0) AS unique_likers,
       COALESCE(ls.follower_like_share,0) AS follower_like_share,
       COALESCE(cs.comments,0) AS comments, COALESCE(cs.unique_commenters,0) AS unique_commenters,
       COALESCE(cs.emoji_comment_share,0) AS emoji_comment_share,
       COALESCE(cs.avg_comment_hashtags,0) AS avg_comment_hashtags,
       COALESCE(ts.tag_count,0) AS tag_count, ts.hashtags
FROM photos p
LEFT JOIN users u ON u.user_id=p.owner_user_id
LEFT JOIN follower_stats fs ON fs.user_id=p.owner_user_id
LEFT JOIN like_stats ls ON ls.photo_id=p.photo_id
LEFT JOIN comment_stats cs ON cs.photo_id=p.photo_id
LEFT JOIN tag_stats ts ON ts.photo_id=p.photo_id
'''
post = pd.read_sql_query(post_sql, conn)
post['engagements'] = post['likes'] + post['comments']
post['engagement_rate'] = np.where(post.owner_followers>0, post.engagements/post.owner_followers, np.nan)
post['comment_to_like_ratio'] = np.where(post.likes>0, post.comments/post.likes, np.nan)
post['active_follower_rate'] = np.where(post.owner_followers>0, post.owner_active_followers/post.owner_followers, np.nan)
post.to_csv(OUTPUT_DIR/'post_analytics.csv', index=False)
print('Analytical table:', post.shape)
display(post.head())

## 7. KPI overview

`Engagements = likes + comments`. The follower-normalized engagement rate is useful for comparing differently sized accounts, but raw engagement is retained because follower counts in this synthetic dataset are unusually similar.

In [ ]:
kpis = pd.Series({
    'users': len(users), 'posts': len(photos), 'likes': len(likes), 'comments': len(comments),
    'total_engagements': int(post.engagements.sum()), 'avg_engagements_per_post': post.engagements.mean(),
    'median_engagements_per_post': post.engagements.median(), 'avg_comments_per_post': post.comments.mean(),
    'posts_with_tags_pct': 100*(post.tag_count>0).mean(), 'active_followers_pct': 100*follows.is_active.mean()
}).round(2)
display(kpis.to_frame('value'))

## 8. Content-type and filter performance

In [ ]:
content_perf = post.groupby('photo_type').agg(posts=('photo_id','count'), avg_likes=('likes','mean'), avg_comments=('comments','mean'), avg_engagements=('engagements','mean'), median_engagements=('engagements','median'), avg_engagement_rate=('engagement_rate','mean')).sort_values('avg_engagements',ascending=False).round(3)
filter_perf = post.groupby('filter_used').agg(posts=('photo_id','count'),avg_engagements=('engagements','mean'),avg_comments=('comments','mean')).rename(index={0:'No filter',1:'Filter'}).round(3)
display(content_perf); display(filter_perf)
fig, axes = plt.subplots(1,2,figsize=(13,4))
sns.barplot(data=content_perf.reset_index(),x='photo_type',y='avg_engagements',ax=axes[0]); axes[0].set_title('Average engagement by content type')
sns.boxplot(data=post,x='photo_type',y='engagements',ax=axes[1]); axes[1].set_title('Engagement distribution by content type')
plt.tight_layout(); plt.show()

## 9. Hashtag analysis

In [ ]:
tag_perf = pd.read_sql_query('''
SELECT t.tag_name, t.location, COUNT(DISTINCT pt.photo_id) AS posts,
       ROUND(AVG(pa.likes + pa.comments),2) AS avg_engagements
FROM photo_tags pt JOIN tags t ON t.tag_id=pt.tag_id
JOIN (''' + post_sql + ''') pa ON pa.photo_id=pt.photo_id
GROUP BY t.tag_id, t.tag_name, t.location
HAVING COUNT(DISTINCT pt.photo_id) >= 5
ORDER BY avg_engagements DESC, posts DESC''', conn)
display(tag_perf.head(10))
plt.figure(figsize=(10,5)); sns.barplot(data=tag_perf.head(10),y='tag_name',x='avg_engagements'); plt.title('Top hashtags by average post engagement (minimum 5 posts)'); plt.tight_layout(); plt.show()

## 10. Follower-growth and community signals

In [ ]:
owner_perf = post.groupby('owner_user_id').agg(posts=('photo_id','count'), followers=('owner_followers','max'), active_followers=('owner_active_followers','max'), avg_engagements=('engagements','mean'), total_engagements=('engagements','sum')).reset_index()
owner_perf['active_follower_rate'] = owner_perf.active_followers/owner_perf.followers.replace(0,np.nan)
display(owner_perf.sort_values(['total_engagements','active_follower_rate'],ascending=False).head(10).round(3))
print('Correlation: active followers vs total engagement =', round(owner_perf[['active_followers','total_engagements']].corr().iloc[0,1],3))
print('Likes from existing followers:', f"{100*likes.is_following.mean():.1f}%")
print('Likes from non-followers (discovery signal):', f"{100*(1-likes.is_following.mean()):.1f}%")

## 11. Posting-time analysis - validity gate

A useful time analysis requires multiple dates and hours. The following cell checks that condition. It intentionally refuses to rank weekdays/hours when the input contains no time variation.

In [ ]:
post['photo_created_at'] = pd.to_datetime(post.photo_created_at, errors='coerce')
enough_time_variation = post.photo_created_at.nunique() >= 10 and post.photo_created_at.dt.date.nunique() >= 3
if enough_time_variation:
    post['weekday'] = post.photo_created_at.dt.day_name()
    post['hour'] = post.photo_created_at.dt.hour
    time_perf = post.groupby(['weekday','hour']).agg(posts=('photo_id','count'),avg_engagements=('engagements','mean')).query('posts >= 3').sort_values('avg_engagements',ascending=False)
    display(time_perf.head(10))
else:
    print('Not enough timestamp variation for a defensible best-time conclusion.')
    print('All supplied post timestamps:', post.photo_created_at.drop_duplicates().astype(str).tolist())

## 12. Data science: predict high-engagement posts

To convert the project from descriptive reporting into decision support, we define **high engagement** as the top quartile of total engagements. Only features available at or before publishing are used; likes/comments are excluded from predictors to avoid target leakage. Class imbalance is handled with `class_weight='balanced'` for Logistic Regression and Random Forest. Models are compared using precision, recall, F1, and ROC-AUC.

In [ ]:
threshold = post.engagements.quantile(.75)
post['high_engagement'] = (post.engagements >= threshold).astype(int)
features = ['photo_type','filter_used','is_private','verified_status','post_count','owner_followers','owner_active_followers','active_follower_rate','tag_count']
X, y = post[features], post.high_engagement
cat_cols=['photo_type']; num_cols=[c for c in features if c not in cat_cols]
preprocess = ColumnTransformer([
    ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols),
    ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_cols)
])
models={
 'Logistic Regression': LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE),
 'Random Forest': RandomForestClassifier(n_estimators=400,min_samples_leaf=4,class_weight='balanced',random_state=RANDOM_STATE),
 'Gradient Boosting': GradientBoostingClassifier(n_estimators=150,max_depth=2,learning_rate=.04,random_state=RANDOM_STATE)
}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
scoring={'precision':'precision','recall':'recall','f1':'f1','roc_auc':'roc_auc'}
rows=[]
for name, model in models.items():
    pipe=Pipeline([('preprocess',preprocess),('model',model)])
    scores=cross_validate(pipe,X,y,cv=cv,scoring=scoring)
    rows.append({'model':name, **{m: scores['test_'+m].mean() for m in scoring}})
model_results=pd.DataFrame(rows).sort_values('roc_auc',ascending=False).reset_index(drop=True)
display(model_results.round(3))
print('Target threshold:',threshold,'engagements | Positive class:',f'{100*y.mean():.1f}%')

## 13. Holdout evaluation and explainability

In [ ]:
best_name=model_results.loc[0,'model']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=RANDOM_STATE)
best_pipe=Pipeline([('preprocess',preprocess),('model',models[best_name])]).fit(X_train,y_train)
pred=best_pipe.predict(X_test); prob=best_pipe.predict_proba(X_test)[:,1]
holdout={'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0),'roc_auc':roc_auc_score(y_test,prob)}
display(pd.Series(holdout,name=best_name).round(3).to_frame())
display(pd.DataFrame(confusion_matrix(y_test,pred),index=['Actual normal','Actual high'],columns=['Pred normal','Pred high']))
RocCurveDisplay.from_predictions(y_test,prob); plt.title(f'ROC curve - {best_name}'); plt.show()

perm=permutation_importance(best_pipe,X_test,y_test,scoring='roc_auc',n_repeats=30,random_state=RANDOM_STATE)
importance=pd.DataFrame({'feature':features,'importance':perm.importances_mean,'std':perm.importances_std}).sort_values('importance',ascending=False)
display(importance.round(4))
plt.figure(figsize=(9,4)); sns.barplot(data=importance,x='importance',y='feature'); plt.title('Feature influence on ROC-AUC (permutation importance)'); plt.tight_layout(); plt.show()

## 14. Business-oriented model interpretation

- A probability is a **prioritization score**, not a guarantee. Use it to decide which planned posts deserve stronger creative review or paid support.
- High recall finds more potentially strong posts; high precision avoids spending resources on false alarms. For campaign planning, choose the threshold based on the cost of missed opportunities versus wasted promotion.
- Feature importance shows association, not causation. Run controlled content experiments before turning an association into a permanent rule.
- This small synthetic dataset has limited timestamp and audience variation. Model performance must be validated on future real Alfido Tech posts before operational use.

## 15. Recommended content calendar (4-week test)

In [ ]:
best_content=content_perf.index[0]
top_tags=tag_perf.head(5).tag_name.tolist()
calendar=pd.DataFrame([
 ['Monday','12:30 PM','Educational carousel','Teach one practical data/AI concept','Saves, comments'],
 ['Wednesday','6:30 PM',f'{best_content.title()} / project proof','Show a result, workflow, or intern success','Shares, profile visits'],
 ['Friday','7:30 PM','Short video / reel','Quick tip with a strong first-2-second hook','Reach, non-follower likes'],
 ['Sunday','11:00 AM','Community post or poll','Ask a focused question and reply quickly','Comments, active followers']
],columns=['Day','Test time (local)','Content','Purpose','Primary KPI'])
display(calendar)
print('Suggested evidence-led hashtag pool:', ', '.join('#'+x.replace(' ','') for x in top_tags))
print('These times are hypotheses for A/B testing because the supplied timestamps cannot identify a historical optimum.')

## 16. Five strategies to increase Alfido Tech engagement

1. **Use a content-mix experiment:** Center the next month on the strongest observed content type, but reserve at least 25% of slots for a challenger format. Compare median engagement, not only averages.
2. **Create controlled hashtag sets:** Combine 2-3 high-performing dataset tags with 2 niche Alfido Tech tags. Rotate sets and compare reach/engagement over at least four comparable posts.
3. **Convert discovery into followers:** Non-follower likes are a discovery signal. Add a single, specific CTA such as “Follow for weekly data-project walkthroughs” and measure profile-to-follow conversion.
4. **Engineer early conversation:** Ask one answerable question in every caption and reply during the first hour. Track comment-to-like ratio and unique commenters, not merely comment volume.
5. **Build a measurement loop:** Record real publish timestamp, reach, impressions, saves, shares, profile visits, follows gained, and content theme. After 30-50 new posts, rerun this notebook to learn genuine best times and validate the model.


## 17. Export decision-ready summary

In [ ]:
summary={
 'dataset': {k:len(v) for k,v in tables.items()},
 'quality': quality,
 'kpis': {k:float(v) for k,v in kpis.items()},
 'best_content_type_by_avg_engagement': str(best_content),
 'top_hashtags_min_5_posts': tag_perf.head(5)[['tag_name','posts','avg_engagements']].to_dict('records'),
 'model_cv_results': model_results.round(4).to_dict('records'),
 'best_model': best_name,
 'holdout_metrics': {k:round(float(v),4) for k,v in holdout.items()},
 'timestamp_limitation': not enough_time_variation
}
with open(OUTPUT_DIR/'analysis_summary.json','w') as f: json.dump(summary,f,indent=2)
conn.close()
print('Created:')
for p in sorted(OUTPUT_DIR.iterdir()): print('-',p)

## Conclusion

This project fulfills the requested Instagram goal using a reproducible data pipeline, a normalized SQLite database, SQL-derived post metrics, content and hashtag analysis, community-growth signals, predictive modeling, and an actionable four-week plan. The most important professional conclusion is also a limitation: the supplied event timestamps have no variation, so any claim about historical best posting times would be fabricated. The recommended schedule is therefore a structured experiment designed to create the missing evidence.